In [1]:
import torch
import time
import psutil
import os
import pandas as pd
import evaluate

from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [3]:
# ==========================
# CONFIGURATION
# ==========================

MODEL_ID = "./finetuned_whisper_model_tiny"

DATASET_ID = "Kennethdot/Ghana_English-Twi_Code-switching_speech"   # change this
SPLIT = "test"

AUDIO_COLUMN = "audio"
TEXT_COLUMN = "text"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ==========================
# LOAD MODEL
# ==========================

print("Loading model...")

processor = WhisperProcessor.from_pretrained(MODEL_ID)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
)

model.to(DEVICE)
model.eval()

print("Device:", DEVICE)


# ==========================
# LOAD DATASET
# ==========================

print("Loading dataset...")

dataset = load_dataset(
    DATASET_ID,
    split=SPLIT
)

dataset = dataset.cast_column(
    AUDIO_COLUMN,
    Audio(sampling_rate=16000)
)

print(dataset)


# ==========================
# METRICS
# ==========================

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


# ==========================
# GPU MONITORING
# ==========================

if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
    

# ==========================
# RUN BENCHMARK
# ==========================

predictions = []
references = []

times = []
cpu_usage = []
ram_usage = []

process = psutil.Process(os.getpid())


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading model...
Device: cuda
Loading dataset...
Dataset({
    features: ['speaker_id', 'age_range', 'gender', 'prompt_set', 'transcript', 'duration', 'split', 'audio', 'file_name', 'error'],
    num_rows: 1731
})


In [4]:
import torch

def transcribe_from_dataset(
    dataset_sample,
    whisper_model,
    processor,
    device="cuda",
    max_new_tokens=128
):
    whisper_model.eval()

    # Extract audio
    audio = dataset_sample["array"]
    sr = dataset_sample["sampling_rate"]

    # Feature extraction (CPU)
    input_features = processor.feature_extractor(
        audio,
        sampling_rate=sr,
        return_tensors="pt"
    ).input_features.to(device)

    # Inference (no grad + faster kernel usage)
    with torch.inference_mode():
        predicted_ids = whisper_model.generate(
            input_features,
            max_new_tokens=max_new_tokens,
            task=TASK,
            forced_decoder_ids=None
        )

    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0].strip()

    return transcription

In [5]:
print("Starting inference...")
for i, sample in enumerate(dataset):

    audio = sample[AUDIO_COLUMN]["array"]

    reference = sample['transcript']

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    inputs = {
        k:v.to(DEVICE)
        for k,v in inputs.items()
    }


    # CPU measurement
    cpu_before = psutil.cpu_percent()


    start = time.perf_counter()

    with torch.inference_mode():
        predicted_ids = model.generate(
            **inputs,
            task="transcribe",
            forced_decoder_ids=None
        )

    end = time.perf_counter()


    cpu_after = psutil.cpu_percent()

    prediction = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]


    predictions.append(prediction)
    references.append(reference)


    times.append(end-start)
    cpu_usage.append(cpu_after)

    ram_usage.append(
        process.memory_info().rss / (1024**2)
    )


    if i % 10 == 0:
        print(
            f"{i}/{len(dataset)} "
            f"time={times[-1]:.2f}s"
        )




Starting inference...
0/1731 time=0.65s
10/1731 time=0.09s
20/1731 time=0.08s
30/1731 time=0.06s
40/1731 time=0.14s
50/1731 time=0.14s
60/1731 time=0.07s
70/1731 time=0.54s
80/1731 time=0.32s
90/1731 time=0.46s
100/1731 time=0.59s
110/1731 time=0.33s
120/1731 time=0.24s
130/1731 time=0.28s
140/1731 time=0.40s
150/1731 time=0.37s
160/1731 time=0.27s
170/1731 time=0.46s
180/1731 time=0.27s
190/1731 time=0.86s
200/1731 time=0.39s
210/1731 time=0.60s
220/1731 time=0.42s
230/1731 time=0.75s
240/1731 time=0.39s
250/1731 time=0.62s
260/1731 time=0.65s
270/1731 time=0.42s
280/1731 time=0.55s
290/1731 time=0.75s
300/1731 time=0.49s
310/1731 time=0.63s
320/1731 time=0.33s
330/1731 time=0.30s
340/1731 time=0.80s
350/1731 time=0.48s
360/1731 time=0.71s
370/1731 time=0.65s
380/1731 time=0.43s
390/1731 time=0.06s
400/1731 time=0.30s
410/1731 time=0.59s
420/1731 time=0.41s
430/1731 time=0.73s
440/1731 time=0.39s
450/1731 time=0.46s
460/1731 time=0.88s
470/1731 time=0.60s
480/1731 time=0.43s
490/1731 

In [10]:
# ==========================
# FINAL METRICS
# ==========================


wer = wer_metric.compute(
    predictions=predictions,
    references=references
)

cer = cer_metric.compute(
    predictions=predictions,
    references=references
)


results = {
    "model": MODEL_ID,
    "WER": wer,
    "CER": cer,
    "avg_inference_time_sec": sum(times)/len(times),
    "avg_cpu_percent": sum(cpu_usage)/len(cpu_usage),
    "peak_ram_MB": max(ram_usage),
}


if DEVICE=="cuda":

    results["peak_gpu_memory_MB"] = (
        torch.cuda.max_memory_allocated()
        / (1024**2)
    )





# ==========================
# SAVE RESULTS
# ==========================

print("\n====== BENCHMARK RESULTS ======")

for k,v in results.items():
    print(f"{k}: {v}")


pd.DataFrame([results]).to_csv(
    "whisper_tiny_baseline_results.csv",
    index=False
)

print("\nSaved: whisper_tiny_baseline_results.csv")


====== BENCHMARK RESULTS ======
model: ./finetuned_whisper_model_tiny
WER: 0.16087785249113146
CER: 0.10297034576104344
avg_inference_time_sec: 0.23392429341094548
avg_cpu_percent: 20.1206239168111
peak_ram_MB: 1681.58203125
peak_gpu_memory_MB: 188.212890625

Saved: whisper_tiny_baseline_results.csv


In [7]:
# Calculate model memory footprint

model_size_bytes = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
)

model_size_mb = model_size_bytes / (1024 ** 2)

print(f"Model size: {model_size_mb:.2f} MB")

Model size: 144.05 MB


In [8]:
print(model.dtype)
print(inputs["input_features"].dtype)
print(inputs["input_features"].shape)

torch.float32
torch.float32
torch.Size([1, 80, 3000])


In [9]:
import numpy as np

durations = np.array(dataset["duration"])

print(f"Number of samples: {len(durations)}")
print(f"Average duration: {durations.mean():.2f} sec")
print(f"Min duration: {durations.min():.2f} sec")
print(f"Max duration: {durations.max():.2f} sec")
print(f"Median duration: {np.median(durations):.2f} sec")
print(f"Std dev: {durations.std():.2f} sec")

Number of samples: 1731
Average duration: 10.07 sec
Min duration: 0.93 sec
Max duration: 51.90 sec
Median duration: 4.85 sec
Std dev: 9.21 sec
